In [13]:
%%capture
!pip install -q numpy==1.26.4
!pip install -q torch==2.3.1 torchvision==0.18.1 --index-url https://download.pytorch.org/whl/cu121
!pip install -q \
    transformers==4.44.2 \
    accelerate==0.33.0 \
    bitsandbytes==0.42.0 \
    triton==2.3.1 \
    sentence-transformers==3.0.1 \
    chromadb==0.5.5 \
    langchain==0.2.16 \
    langchain-community==0.2.16 \
    langchain-huggingface==0.0.3 \
    pypdf==4.3.1 \
    nltk==3.8.1 \
    rouge-score==0.1.2 \
    bert-score==0.3.13 \
    ragas==0.1.14 \
    pandas matplotlib seaborn tqdm
import IPython
IPython.Application.instance().kernel.do_shutdown(True)

In [1]:
!pip install -q streamlit
!npm install -g localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 83.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 125.7 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧
added 22 packages in 2s
⠧
⠧3 packages are looking for funding
⠧  run `npm fund` for details
⠧

In [11]:
%%writefile app.py
import streamlit as st
import os
import re
import time
import textwrap
import warnings
from pathlib import Path
import numpy as np
import torch
import nltk
from pypdf import PdfReader
from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForCausalLM
import chromadb

warnings.filterwarnings('ignore')

DEVICE   = 'cuda' if torch.cuda.is_available() else 'cpu'
DTYPE    = torch.float16 if DEVICE == 'cuda' else torch.float32
CHROMA_PATH = '/content/chroma_db'
CHUNK_SIZE  = 512
CHUNK_OVERLAP = 64
TOP_K       = 4

EMBED_MODEL_ID  = 'sentence-transformers/all-MiniLM-L6-v2'
LLM_MODEL_ID = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
RERANKER_ID     = 'cross-encoder/ms-marco-MiniLM-L-6-v2'

@st.cache_resource
def load_models_and_client():
    nltk.download('punkt', quiet=True)
    nltk.download('punkt_tab', quiet=True)

    embed = SentenceTransformer(EMBED_MODEL_ID, device=DEVICE)
    rerank = CrossEncoder(RERANKER_ID, device=DEVICE)

    tok = AutoTokenizer.from_pretrained(LLM_MODEL_ID)
    llm = AutoModelForCausalLM.from_pretrained(
        LLM_MODEL_ID,
        torch_dtype=DTYPE,
        low_cpu_mem_usage=True,
    ).to(DEVICE)
    llm.eval()

    client = chromadb.PersistentClient(path=CHROMA_PATH)
    return embed, rerank, tok, llm, client

embed_model, reranker, tokenizer, llm_model, chroma_client = load_models_and_client()

if 'CONVERSATION_HISTORY' not in st.session_state:
    st.session_state.CONVERSATION_HISTORY = []
if 'messages' not in st.session_state:
    st.session_state.messages = []
if 'status_msg' not in st.session_state:
    st.session_state.status_msg = ""

def reset_collection(name='rag_docs'):
    try:
        chroma_client.delete_collection(name)
    except Exception:
        pass
    return chroma_client.create_collection(name=name, metadata={'hnsw:space': 'cosine'})

def extract_text_from_pdf(pdf_path: str) -> list:
    reader = PdfReader(pdf_path)
    pages = []
    for i, page in enumerate(reader.pages):
        text = page.extract_text() or ''
        text = re.sub(r'\s+', ' ', text).strip()
        if text:
            pages.append({'page': i + 1, 'text': text})
    return pages

def chunk_text(text: str, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP) -> list:
    words = text.split()
    chunks, i = [], 0
    while i < len(words):
        chunk = ' '.join(words[i: i + chunk_size])
        if chunk:
            chunks.append(chunk)
        i += chunk_size - overlap
    return chunks

def ingest_pdf(pdf_path: str, source_name: str = None) -> dict:
    collection = reset_collection()
    name = source_name or Path(pdf_path).name

    pages = extract_text_from_pdf(pdf_path)
    all_chunks, all_meta = [], []
    for pg in pages:
        for chunk in chunk_text(pg['text']):
            all_chunks.append(chunk)
            all_meta.append({'source': name, 'page': pg['page']})

    t0 = time.time()
    embeddings = embed_model.encode(
        all_chunks, batch_size=64, show_progress_bar=False, normalize_embeddings=True
    ).tolist()
    embed_time = time.time() - t0

    ids = [f'chunk_{i}' for i in range(len(all_chunks))]
    collection.add(documents=all_chunks, embeddings=embeddings, metadatas=all_meta, ids=ids)

    stats = {
        'file': name,
        'pages': len(pages),
        'chunks': len(all_chunks),
        'embed_time_s': round(embed_time, 2),
        'avg_chunk_words': round(np.mean([len(c.split()) for c in all_chunks]), 1) if all_chunks else 0,
    }
    return stats, collection

def retrieve(query: str, collection, top_k: int = TOP_K, rerank: bool = True) -> list:
    if collection is None or collection.count() == 0:
        return []

    q_emb = embed_model.encode([query], normalize_embeddings=True).tolist()

    results = collection.query(
        query_embeddings=q_emb,
        n_results=min(top_k * 3, collection.count()),
        include=['documents', 'metadatas', 'distances'],
    )

    docs  = results['documents'][0]
    metas = results['metadatas'][0]
    dists = results['distances'][0]

    candidates = [
        {'text': d, 'meta': m, 'cosine_score': round(1 - dist, 4)}
        for d, m, dist in zip(docs, metas, dists)
    ]

    if rerank and len(candidates) > 1:
        pairs  = [(query, c['text']) for c in candidates]
        scores = reranker.predict(pairs)
        for c, s in zip(candidates, scores):
            c['rerank_score'] = round(float(s), 4)
        candidates.sort(key=lambda x: x['rerank_score'], reverse=True)
        for c in candidates:
            c['rerank_score'] = round(float(torch.sigmoid(torch.tensor(c['rerank_score'])).item()), 4)

    return candidates[:top_k]

def generate_answer(query: str, chunks: list) -> dict:
    t0 = time.time()

    context_str = '\n'.join([c['text'] for c in chunks])

    prompt = (
        f"<|system|>\nYou are a factual assistant. Answer the question using only the context block below. "
        f"If the answer is missing from the text context, say 'I could not find this in the document.'\n"
        f"CONTEXT:\n{context_str}</s>\n"
        f"<|user|>\nQuestion: {query}</s>\n"
        f"<|assistant|>\n"
    )

    inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=2048).to(DEVICE)
    input_len = inputs['input_ids'].shape[1]  # Extracted the explicit column dimension integer

    with torch.no_grad():
        outputs = llm_model.generate(
            **inputs,
            max_new_tokens=128,
            temperature=0.01,   # Forced zero-creativity to stop repeating patterns
            do_sample=False,
            repetition_penalty=1.2
        )

    new_tokens = outputs[0, input_len:]
    answer = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

    if not answer or answer.isspace():
        answer = "I could not find this in the document."

    latency = round(time.time() - t0, 2)
    return {
        'answer': answer,
        'reasoning': '',
        'latency_s': latency,
        'tokens_generated': len(new_tokens)
    }

def rag_query(query: str, collection, use_history: bool = True) -> dict:
    full_query = query
    if use_history and st.session_state.CONVERSATION_HISTORY:
        last_turns = st.session_state.CONVERSATION_HISTORY[-2:]
        history_str = '\n'.join([f"User: {t['query']}\nAssistant: {t['answer']}" for t in last_turns])
        full_query = f"Previous conversation Context:\n{history_str}\n\nNew user query: {query}"

    chunks = retrieve(full_query, collection, top_k=TOP_K)
    if not chunks:
        return {'answer': "Please ingest a document before querying.", 'reasoning': '', 'chunks': []}

    result = generate_answer(query, chunks) # Send raw clean query to stop history bleed
    st.session_state.CONVERSATION_HISTORY.append({'query': query, 'answer': result['answer']})
    result['chunks'] = chunks
    return result

st.set_page_config(page_title="RAG Chatbot — DeepSeek", layout="wide")

st.markdown("# RAG Chatbot")
st.markdown("Upload a PDF, then ask questions about its contents.")

with st.sidebar:
    st.header("Document Ingestion")
    pdf_file = st.file_uploader("Upload PDF", type=["pdf"])
    upload_btn = st.button("Ingest PDF", type="primary", use_container_width=True)

    if upload_btn:
        if pdf_file is None:
            st.session_state.status_msg = "No file uploaded."
        else:
            temp_path = os.path.join("/tmp", pdf_file.name)
            with open(temp_path, "wb") as f:
                f.write(pdf_file.getbuffer())

            st.session_state.messages = []
            st.session_state.CONVERSATION_HISTORY.clear()

            try:
                with st.spinner("Processing embeddings..."):
                    stats, active_coll = ingest_pdf(temp_path)
                    st.session_state.active_collection_name = 'rag_docs'
                    st.session_state.status_msg = (
                        f"**{stats['file']}** ingested!\n\n"
                        f"• Pages: {stats['pages']}  \n• Chunks: {stats['chunks']}  \n"
                        f"• Embed time: {stats['embed_time_s']}s"
                    )
            except Exception as e:
                st.session_state.status_msg = f"Error ingesting file: {e}"

    if st.session_state.status_msg:
        st.info(st.session_state.status_msg)

chat_container = st.container()

with chat_container:
    for msg in st.session_state.messages:
        with st.chat_message(msg["role"]):
            st.markdown(msg["content"])

if user_message := st.chat_input("Ask something about the document…"):
    with chat_container:
        with st.chat_message("user"):
            st.markdown(user_message)
    st.session_state.messages.append({"role": "user", "content": user_message})

    try:
        current_coll = chroma_client.get_collection(name='rag_docs') if 'active_collection_name' in st.session_state else None
        r = rag_query(user_message, current_coll)
        bot_response = r.get('answer', 'No response key found.')

        chunks = r.get('chunks', [])
        if chunks:
            sources = '\n'.join(
                [f"• p.{c['meta']['page']} — score {c.get('rerank_score', c.get('cosine_score', 0)):.3f}"
                 for c in chunks if 'meta' in c]
            )
            bot_response += f"\n\n**Sources:**\n{sources}"
    except Exception as e:
        bot_response = f"Error: {e}"

    with chat_container:
        with st.chat_message("assistant"):
            st.markdown(bot_response)
    st.session_state.messages.append({"role": "assistant", "content": bot_response})

Overwriting app.py


In [12]:
import time
from google.colab import output

get_ipython().system_raw('streamlit run app.py --server.port 8501 --server.enableCORS=false --server.enableXsrfProtection=false &')
time.sleep(5)
output.serve_kernel_port_as_iframe(8501, height=800)


<IPython.core.display.Javascript object>